# 🎙️ Indian Accent Detector — Full Training Pipeline

This notebook runs the complete accent classification pipeline on a GPU runtime.

## Research Gaps This Project Closes

1. **Short-clip benchmarking**: We benchmark accuracy across 1s, 2s, and 3s clips. Existing literature (e.g., MPSA-DenseNet, AccentDB) only evaluates on longer clips, ignoring real-world scenarios where only short snippets are available.

2. **Hierarchical Indian sub-accent classification**: We use the Svarah dataset to classify Indian English into 4 sub-regions (North, South, East, West), going beyond the single "Indian" label used in prior work.

3. **Standardized multi-metric evaluation**: We report per-class F1, normalized confusion matrices, and clip-length curves on a single reproducible public split — something no prior accent classification paper provides.

## Requirements
- **GPU**: Kaggle P100 or Colab T4 (free tier works)
- **Runtime**: ~2-3 hours for full 3-length training
- **Disk**: ~10 GB for datasets + models

---
## 0. Setup & Install Dependencies

In [ ]:
# Install all dependencies
!pip install -q transformers>=4.38.0 datasets>=2.18.0 torch>=2.1.0 torchaudio>=2.1.0 \
    soundfile librosa scikit-learn matplotlib seaborn gradio>=4.0.0 \
    pandas numpy huggingface_hub evaluate accelerate>=0.21.0

In [ ]:
# GPU check — this notebook REQUIRES a GPU
import torch
assert torch.cuda.is_available(), "❌ No GPU detected! Enable GPU in Runtime > Change runtime type."
print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Clone the repo (or upload files manually)
# If running from the repo directory, skip this cell
import os
if not os.path.exists('config.py'):
    print("config.py not found — please upload project files or clone the repo.")
    print("Example: !git clone https://github.com/YOUR_USERNAME/accent_detector.git")
    print("Then: %cd accent_detector")
else:
    print("✅ Project files found.")

In [ ]:
# Set random seeds for reproducibility
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

---
## 1. Data Preparation

This step:
- Downloads Common Voice English (filtered to 4 global accents)
- Attempts to load Svarah for Indian sub-accents (falls back to 4-class if unavailable)
- Creates 80/10/10 stratified splits
- Generates 3 clip-length variants (1s, 2s, 3s)
- Saves a reproducible split manifest

In [ ]:
# Optional: Login to HuggingFace for gated datasets (Svarah, Common Voice)
# Uncomment and run if you have a HF token:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
# Run data preparation
!python prepare_data.py

In [ ]:
# Verify processed data exists
import os
for clip_len in [1, 2, 3]:
    path = f"processed_data/clips_{clip_len}s"
    if os.path.exists(path):
        print(f"✅ {path} exists")
    else:
        print(f"❌ {path} missing!")

# Check manifest
manifest_path = "processed_data/split_manifest.csv"
if os.path.exists(manifest_path):
    import pandas as pd
    manifest = pd.read_csv(manifest_path)
    print(f"\n✅ Manifest: {len(manifest)} entries")
    print(manifest['split'].value_counts())
    print(manifest['accent'].value_counts())
else:
    print(f"❌ Manifest missing!")

---
## 2. Model Training

Trains Wav2Vec2ForSequenceClassification for the 3s clip length.
- CNN feature encoder is frozen
- Cosine LR schedule with 10% warmup
- Best model selected by macro F1 (not accuracy)

To train all clip lengths, change `--clip_length 3` to `--all`.

In [ ]:
# Train for 3s clips (recommended for notebook — faster than --all)
!python train.py --clip_length 3

In [ ]:
# Optional: Train for all clip lengths (takes ~3x longer)
# !python train.py --all

In [ ]:
# Verify model was saved
import os
model_path = "accent-classifier-final/clips_3s"
if os.path.exists(model_path):
    files = os.listdir(model_path)
    print(f"✅ Model saved at {model_path}")
    print(f"   Files: {files}")
else:
    print(f"❌ Model not found at {model_path}")

---
## 3. Evaluation

Runs the full evaluation pipeline:
- Per-class precision/recall/F1 CSV
- Normalized confusion matrix (PNG + CSV)
- Overall metrics CSV
- Clip-length accuracy curve (if multiple models trained)

In [ ]:
# Run evaluation
!python evaluate.py --clip_length 3

In [ ]:
# Display results
import pandas as pd
from IPython.display import display, Image

# Per-class metrics
per_class_path = "results/per_class_3s.csv"
if os.path.exists(per_class_path):
    print("Per-Class Metrics (3s):")
    display(pd.read_csv(per_class_path))

# Overall metrics
overall_path = "results/overall_metrics.csv"
if os.path.exists(overall_path):
    print("\nOverall Metrics:")
    display(pd.read_csv(overall_path))

# Confusion matrix
cm_path = "results/confusion_matrix_3s.png"
if os.path.exists(cm_path):
    print("\nConfusion Matrix:")
    display(Image(cm_path))

---
## 4. Launch Gradio Demo

A web interface where you can record audio or upload files to get accent predictions.

In [ ]:
# Launch Gradio demo with a public share link
!python app.py --share

---
## 5. Save Results

Download the trained model and results for later use.

In [ ]:
# List all generated artifacts
import os

print("Generated files:")
for root, dirs, files in os.walk('results'):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"  {fpath} ({size:,} bytes)")

print("\nModel files:")
for root, dirs, files in os.walk('accent-classifier-final'):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"  {fpath} ({size:,} bytes)")

In [ ]:
# Optional: Zip results for download
!zip -r accent_detector_results.zip results/ accent-classifier-final/
print("\n✅ Results zipped to accent_detector_results.zip")